In [2]:
# ============================================================
# Simulador de Sincronización Discreta tipo Kuramoto sobre Grafos
# ============================================================
#
# CORRECCIONES APLICADAS (respecto al borrador anterior):
#
#   1. sigma_kappa: umbral incluyente (>= y <=) conforme al paper.
#      La versión anterior usaba > y <, lo que impedía reproducir
#      el ejemplo (0,2,3) -> (3,2,3) -> (3,3,3).
#
#   2. dM_signed(a, b): reescrita con el orden natural a - b,
#      de modo que el nombre de argumentos coincide directamente
#      con la fórmula del paper d_M(a, b) = a - b (mod M, centrada).
#      S_i llama dM_signed(theta[j], theta[i]) sin inversión.
#
#   3. subred_acuerdo_edges: usa < epsilon (estricto), no <=.
#      Con epsilon=1 esto garantiza acuerdo exacto (distancia 0),
#      que es la convención del paper.
#      El widget ofrece un selector "estricto / incluyente" para uso
#      didáctico.
#
# ============================================================

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output
from itertools import product


# ============================================================
# 1. Funciones auxiliares
# ============================================================

def bell_number(n):
    """
    Número de Bell B_n.
    Cuenta el número de particiones de un conjunto con n elementos.
    """
    bell = [[0] * (n + 1) for _ in range(n + 1)]
    bell[0][0] = 1
    for i in range(1, n + 1):
        bell[i][0] = bell[i - 1][i - 1]
        for j in range(1, i + 1):
            bell[i][j] = bell[i - 1][j - 1] + bell[i][j - 1]
    return bell[n][0]


def kuramoto_order_parameter(theta, M):
    """
    Parámetro de orden de Kuramoto para fases discretas:

        r(t) = |(1/N) sum_j exp(2 pi i theta_j / M)|

    r = 1  ->  sincronización perfecta.
    r ~ 0  ->  fases dispersas.
    """
    z = np.exp(2j * np.pi * np.array(theta, dtype=int) / M)
    return float(np.abs(np.mean(z)))


# ============================================================
# 2. Clase principal del modelo
# ============================================================

class KuramotoDiscreto:
    """
    Red discreta de Kuramoto con umbral sobre un grafo finito.

    Modelo (conforme al paper):

        (F_kappa(theta))_i = theta_i + omega* + sigma_kappa(S_i(theta))  mod M

    donde:

        S_i(theta) = sum_{j in N(i)} d_M(theta_j, theta_i)

        d_M(a, b)  = ((a - b + floor(M/2)) mod M) - floor(M/2)

        sigma_kappa(S) = +1  si  S >= kappa
                       = -1  si  S <= -kappa
                       =  0  si  |S| < kappa

    Parámetro epsilon_modo:
        "estricto"   -> subred usa d_M < epsilon  (convención del paper)
        "incluyente" -> subred usa d_M <= epsilon (conveniente para didáctica)
    """

    def __init__(
        self,
        N=4,
        M=6,
        kappa=2,
        omega=0,
        tipo_grafo="Completo",
        epsilon=1,
        epsilon_modo="estricto",
    ):
        self.N = int(N)
        self.M = int(M)
        self.kappa = int(kappa)
        self.omega = int(omega)
        self.tipo_grafo = tipo_grafo
        self.epsilon = float(epsilon)
        self.epsilon_modo = epsilon_modo   # "estricto" | "incluyente"
        self.G = self._crear_grafo()
        self.pos = nx.spring_layout(self.G, seed=7)

    # --------------------------------------------------------
    # Grafo
    # --------------------------------------------------------
    def _crear_grafo(self):
        tipos = {
            "Completo": nx.complete_graph(self.N),
            "Camino":   nx.path_graph(self.N),
            "Ciclo":    nx.cycle_graph(self.N),
            "Estrella": nx.star_graph(self.N - 1),
        }
        if self.tipo_grafo not in tipos:
            raise ValueError(
                f"Tipo de grafo '{self.tipo_grafo}' no reconocido. "
                "Usa: Completo, Camino, Ciclo o Estrella."
            )
        return tipos[self.tipo_grafo]

    # --------------------------------------------------------
    # CORRECCIÓN 2 — dM_signed con orden natural a - b
    # --------------------------------------------------------
    def dM_signed(self, a, b):
        """
        Diferencia angular con signo (conforme al paper):

            d_M(a, b) = ((a - b + floor(M/2)) mod M) - floor(M/2)

        Mide el desplazamiento orientado desde b hacia a.
        Rango: [ -floor(M/2),  floor(M/2) ]  (asimétrico en M par).
        """
        return ((int(a) - int(b) + self.M // 2) % self.M) - self.M // 2

    def dM_circular(self, a, b):
        """
        Distancia circular sin signo:

            d_M(a, b) = min(|a - b|, M - |a - b|)

        Usada para construir la subred epsilon-sincronizada.
        """
        diff = abs(int(a) - int(b))
        return min(diff, self.M - diff)

    # --------------------------------------------------------
    # CORRECCIÓN 1 — sigma_kappa con umbral incluyente
    # --------------------------------------------------------
    def sigma_kappa(self, S):
        """
        Función signo umbralizada (conforme al paper):

            sigma_kappa(S) = +1  si  S >= kappa
                           = -1  si  S <= -kappa
                           =  0  si  |S| < kappa

        El umbral es INCLUYENTE: el vértice ajusta su fase
        cuando el desbalance *alcanza* el umbral kappa.
        """
        if S >= self.kappa:
            return 1
        elif S <= -self.kappa:
            return -1
        else:
            return 0

    # --------------------------------------------------------
    # CORRECCIÓN 2 (continuación) — S_i con dM_signed(theta[j], theta[i])
    # --------------------------------------------------------
    def S_i(self, theta, i):
        """
        Suma de acoplamiento con signo del vértice i:

            S_i(theta) = sum_{j in N(i)} d_M(theta_j, theta_i)

        El argumento a = theta[j] (vecino), b = theta[i] (yo).
        """
        theta = np.array(theta, dtype=int)
        return sum(
            self.dM_signed(theta[j], theta[i])
            for j in self.G.neighbors(i)
        )

    def vector_S(self, theta):
        """Vector S(theta) = (S_1, ..., S_N)."""
        return np.array([self.S_i(theta, i) for i in range(self.N)], dtype=int)

    def vector_sigma(self, theta):
        """Vector de correcciones sigma_kappa(S_i(theta))."""
        return np.array([self.sigma_kappa(s) for s in self.vector_S(theta)], dtype=int)

    # --------------------------------------------------------
    # Dinámica F_kappa
    # --------------------------------------------------------
    def paso(self, theta):
        """Una iteración de F_kappa."""
        theta = np.array(theta, dtype=int)
        return (theta + self.omega + self.vector_sigma(theta)) % self.M

    def simular(self, theta0, T=30):
        """
        Trayectoria theta^0, theta^1, ..., theta^T.

        Nota sobre índices:
          - trayectoria[t]  y  r_hist[t]   corresponden a theta en el paso t.
          - S_hist[t]  y  sigma_hist[t]    son los valores calculados EN t
            que producen trayectoria[t+1].  Por eso tienen longitud T, no T+1.
          - El último renglón de la tabla (t = T) muestra '-' en S_i y sigma,
            porque no se aplica un paso adicional.
        """
        theta = np.array(theta0, dtype=int) % self.M
        trayectoria = [theta.copy()]
        S_hist, sigma_hist = [], []
        r_hist = [kuramoto_order_parameter(theta, self.M)]
        subredes = [self._subred_edges(theta)]

        for _ in range(T):
            S_t = self.vector_S(theta)
            sig_t = self.vector_sigma(theta)
            S_hist.append(S_t.copy())
            sigma_hist.append(sig_t.copy())

            theta = self.paso(theta)
            trayectoria.append(theta.copy())
            r_hist.append(kuramoto_order_parameter(theta, self.M))
            subredes.append(self._subred_edges(theta))

        return {
            "trayectoria": np.array(trayectoria),
            "S_hist":      np.array(S_hist),
            "sigma_hist":  np.array(sigma_hist),
            "r_hist":      np.array(r_hist),
            "subredes":    subredes,
        }

    # --------------------------------------------------------
    # CORRECCIÓN 3 — subred epsilon-sincronizada con < epsilon
    # --------------------------------------------------------
    def _subred_edges(self, theta):
        """
        Aristas de la subred epsilon-sincronizada.

        Convención según epsilon_modo:
          "estricto"   ->  d_M(theta_i, theta_j) <  epsilon  (paper)
          "incluyente" ->  d_M(theta_i, theta_j) <= epsilon  (didáctico)

        Con epsilon=1 y modo estricto sólo hay acuerdo exacto (distancia 0).
        """
        theta = np.array(theta, dtype=int)
        estricto = (self.epsilon_modo == "estricto")
        edges = []
        for i, j in self.G.edges():
            d = self.dM_circular(theta[i], theta[j])
            if (d < self.epsilon) if estricto else (d <= self.epsilon):
                edges.append((i, j))
        return tuple(sorted(edges))

    # Alias público para compatibilidad con código existente
    def subred_acuerdo_edges(self, theta):
        return self._subred_edges(theta)

    # --------------------------------------------------------
    # Sincronización
    # --------------------------------------------------------
    def esta_sincronizado(self, theta):
        """¿theta pertenece a la diagonal Delta_M?"""
        theta = np.array(theta, dtype=int)
        return bool(np.all(theta == theta[0]))

    # --------------------------------------------------------
    # Clasificación de atractores
    # --------------------------------------------------------
    def clasificar_atractor(self, theta0, T_max=200):
        """
        Clasifica la trayectoria como:

          "sync"                 ->  todos los vértices con la misma fase.
          "punto fijo no trivial"->  punto fijo pero no sincronizado.
          "ciclo"                ->  ciclo de periodo > 1.
          "transitorio"          ->  no converge en T_max pasos.

        Nota: si omega != 0, el sistema puede sincronizarse en un estado
        que rota (todos iguales, pero cambiando de valor en cada paso).
        En ese caso se devuelve "sync" con el periodo real del ciclo rotante,
        calculado como M / gcd(omega + sigma, M).  Para omega=0 el periodo es 1.
        """
        theta = np.array(theta0, dtype=int) % self.M
        vistos = {}

        for t in range(T_max + 1):
            key = tuple(theta.tolist())

            if self.esta_sincronizado(theta):
                # Detecta si es fijo o rotante
                theta_next = self.paso(theta)
                if self.esta_sincronizado(theta_next) and np.all(theta_next == theta):
                    periodo = 1
                else:
                    # Busca el periodo del ciclo rotante (máx M pasos)
                    periodo = 1
                    tmp = theta_next.copy()
                    for p in range(1, self.M + 1):
                        if np.all(tmp == theta):
                            periodo = p
                            break
                        tmp = self.paso(tmp)
                return {
                    "tipo":         "sync",
                    "tiempo":       t,
                    "periodo":      periodo,
                    "estado_final": theta.copy(),
                }

            if key in vistos:
                t_inicio = vistos[key]
                periodo = t - t_inicio
                tipo = "punto fijo no trivial" if periodo == 1 else "ciclo"
                return {
                    "tipo":         tipo,
                    "tiempo":       t_inicio,
                    "periodo":      periodo,
                    "estado_final": theta.copy(),
                }

            vistos[key] = t
            theta = self.paso(theta)

        return {
            "tipo":         "transitorio",
            "tiempo":       T_max,
            "periodo":      None,
            "estado_final": theta.copy(),
        }

    # --------------------------------------------------------
    # Tabla de trayectoria
    # --------------------------------------------------------
    def tabla_trayectoria(self, resultado):
        trayectoria = resultado["trayectoria"]
        S_hist      = resultado["S_hist"]
        sigma_hist  = resultado["sigma_hist"]
        r_hist      = resultado["r_hist"]
        subredes    = resultado["subredes"]
        T           = len(trayectoria) - 1

        filas = []
        for t in range(T + 1):
            fila = {
                "t":              t,
                "theta(t)":       tuple(trayectoria[t]),
                "r(t)":           round(r_hist[t], 4),
                "subred_acuerdo": subredes[t],
            }
            if t < T:
                fila["S_i(theta_t)"]      = tuple(S_hist[t])
                fila["sigma_kappa(S_i)"]  = tuple(sigma_hist[t])
            else:
                fila["S_i(theta_t)"]      = "-"
                fila["sigma_kappa(S_i)"]  = "-"
            filas.append(fila)

        return pd.DataFrame(filas)


# ============================================================
# 3. Exploración exhaustiva del espacio X_M = Z_M^V
# ============================================================

def exploracion_exhaustiva(
    N, M, kappa, omega=0, tipo_grafo="Completo",
    epsilon=1, epsilon_modo="estricto", T_max=150
):
    """
    Recorre todo el espacio M^N y calcula proporciones de regímenes,
    tiempos de sincronización, subredes realizables y transiciones.
    """
    modelo = KuramotoDiscreto(
        N=N, M=M, kappa=kappa, omega=omega,
        tipo_grafo=tipo_grafo, epsilon=epsilon, epsilon_modo=epsilon_modo,
    )

    total = M ** N
    conteo = {"sync": 0, "punto fijo no trivial": 0, "ciclo": 0, "transitorio": 0}
    tiempos_sync = []
    subredes_realizables = set()
    transiciones = set()

    for theta0 in product(range(M), repeat=N):
        # Una sola simulación por configuración
        resultado = modelo.simular(theta0, T=min(T_max, 50))
        clasif    = modelo.clasificar_atractor(theta0, T_max=T_max)

        conteo[clasif["tipo"]] += 1
        if clasif["tipo"] == "sync":
            tiempos_sync.append(clasif["tiempo"])

        subredes = resultado["subredes"]
        for s in subredes:
            subredes_realizables.add(s)
        for a, b in zip(subredes[:-1], subredes[1:]):
            if a != b:
                transiciones.add((a, b))

    return {
        "total_configuraciones":  total,
        "sync":                   conteo["sync"],
        "fix":                    conteo["punto fijo no trivial"],
        "cyc":                    conteo["ciclo"],
        "transitorio":            conteo["transitorio"],
        "rho_sync":               conteo["sync"] / total,
        "rho_fix":                conteo["punto fijo no trivial"] / total,
        "rho_cyc":                conteo["ciclo"] / total,
        "rho_transitorio":        conteo["transitorio"] / total,
        "tiempos_sync":           tiempos_sync,
        "num_subredes_realizables": len(subredes_realizables),
        "num_transiciones":       len(transiciones),
        "Bell_BN":                bell_number(N),
        "subredes_realizables":   subredes_realizables,
        "transiciones":           transiciones,
    }


# ============================================================
# 4. Barrido de kappa para mapa de regímenes
# ============================================================

def barrido_kappa(
    N, M, omega=0, tipo_grafo="Completo",
    epsilon=1, epsilon_modo="estricto", T_max=150
):
    """
    Barre kappa desde 1 hasta degmax * floor(M/2).
    Devuelve DataFrame con rho_sync, rho_fix, rho_cyc por kappa.
    """
    modelo_ref = KuramotoDiscreto(N=N, M=M, kappa=1, tipo_grafo=tipo_grafo)
    degmax     = max(dict(modelo_ref.G.degree()).values())
    kappa_max  = degmax * (M // 2)

    filas = []
    for kappa in range(1, kappa_max + 1):
        res = exploracion_exhaustiva(
            N=N, M=M, kappa=kappa, omega=omega,
            tipo_grafo=tipo_grafo, epsilon=epsilon,
            epsilon_modo=epsilon_modo, T_max=T_max,
        )
        filas.append({
            "kappa":               kappa,
            "rho_sync":            res["rho_sync"],
            "rho_fix":             res["rho_fix"],
            "rho_cyc":             res["rho_cyc"],
            "rho_transitorio":     res["rho_transitorio"],
            "subredes_realizables": res["num_subredes_realizables"],
            "transiciones":        res["num_transiciones"],
            "Bell_BN":             res["Bell_BN"],
        })
    return pd.DataFrame(filas)


# ============================================================
# 5. Visualizaciones
# ============================================================

def plot_heatmap(resultado):
    trayectoria = resultado["trayectoria"]
    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(trayectoria.T, aspect="auto", interpolation="nearest")
    ax.set_xlabel("Tiempo t")
    ax.set_ylabel("Vértice i")
    ax.set_title("Heatmap de evolución de fases")
    plt.colorbar(im, ax=ax, label="Fase en Z_M")
    plt.tight_layout()
    plt.show()


def plot_order_parameter(resultado):
    r = resultado["r_hist"]
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(range(len(r)), r, marker="o")
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel("Tiempo t")
    ax.set_ylabel("r(t)")
    ax.set_title("Parámetro de orden de Kuramoto")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_grafo(modelo, theta, title="Grafo coloreado por fase"):
    theta = np.array(theta, dtype=int)
    fig, ax = plt.subplots(figsize=(5, 5))

    nx.draw_networkx_edges(modelo.G, modelo.pos, ax=ax, alpha=0.25)

    acuerdo = modelo.subred_acuerdo_edges(theta)
    if acuerdo:
        nx.draw_networkx_edges(
            modelo.G, modelo.pos, edgelist=acuerdo, ax=ax, width=3
        )

    nodes = nx.draw_networkx_nodes(
        modelo.G, modelo.pos,
        node_color=theta.astype(float),   # float para evitar advertencias de matplotlib
        cmap=plt.cm.viridis,
        vmin=0, vmax=modelo.M - 1,
        node_size=650, ax=ax,
    )
    nx.draw_networkx_labels(modelo.G, modelo.pos, ax=ax)
    plt.colorbar(nodes, ax=ax, label="Fase")
    ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


def plot_hist_tiempos(tiempos):
    if not tiempos:
        print("No hubo trayectorias sincronizadas; no hay histograma de tiempos.")
        return
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.hist(tiempos, bins=range(min(tiempos), max(tiempos) + 2), align="left", rwidth=0.85)
    ax.set_xlabel("Tiempo de sincronización")
    ax.set_ylabel("Frecuencia")
    ax.set_title("Histograma de tiempos de sincronización")
    plt.tight_layout()
    plt.show()


def plot_mapa_regimenes(df):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(df["kappa"], df["rho_sync"], marker="o", label="rho_sync")
    ax.plot(df["kappa"], df["rho_fix"],  marker="o", label="rho_fix")
    ax.plot(df["kappa"], df["rho_cyc"],  marker="o", label="rho_cyc")
    ax.set_xlabel("kappa")
    ax.set_ylabel("Proporción")
    ax.set_title("Mapa de regímenes por barrido de kappa")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()


def construir_diagrama_transiciones(transiciones):
    D = nx.DiGraph()
    for a, b in transiciones:
        D.add_node(a)
        D.add_node(b)
        D.add_edge(a, b)
    return D


def plot_diagrama_transiciones(transiciones, max_labels=20):
    D = construir_diagrama_transiciones(transiciones)
    if D.number_of_nodes() == 0:
        print("No hay transiciones no triviales entre subredes.")
        return
    fig, ax = plt.subplots(figsize=(8, 6))
    pos = nx.spring_layout(D, seed=11)
    nx.draw_networkx_nodes(D, pos, node_size=500, ax=ax)
    nx.draw_networkx_edges(D, pos, arrows=True, arrowstyle="->", arrowsize=15, ax=ax)
    if D.number_of_nodes() <= max_labels:
        nx.draw_networkx_labels(D, pos, labels={n: str(n) for n in D.nodes()}, font_size=7, ax=ax)
    ax.set_title("Diagrama de transiciones de subredes")
    ax.axis("off")
    plt.tight_layout()
    plt.show()


# ============================================================
# 6. Conclusión de sincronización
# ============================================================

def imprimir_conclusion_simulacion(modelo, resultado, clasif, T):
    """
    Imprime una conclusión legible sobre si la subred sincronizó o no,
    con detalles del estado final y el parámetro de orden.

    Se usa al final de la simulación individual.
    """
    sep  = "=" * 50
    sep2 = "-" * 50
    tipo = clasif["tipo"]
    r_final = float(resultado["r_hist"][-1])
    r_ini   = float(resultado["r_hist"][0])
    theta_final = tuple(clasif["estado_final"])

    print()
    print(sep)
    print("  CONCLUSIÓN")
    print(sep)

    if tipo == "sync":
        t_sync  = clasif["tiempo"]
        periodo = clasif["periodo"]
        fase    = theta_final[0]

        if periodo == 1:
            print("  ✔  LA SUBRED SINCRONIZÓ (punto fijo global)")
            print(sep2)
            print(f"  Todos los vértices alcanzaron la fase {fase} en t = {t_sync}.")
            print(f"  El sistema permanece en θ = {theta_final} para todo t ≥ {t_sync}.")
        else:
            print("  ✔  LA SUBRED SINCRONIZÓ (rotación global)")
            print(sep2)
            print(f"  Todos los vértices coinciden en fase y rotan juntos.")
            print(f"  Periodo del ciclo sincronizado: {periodo} pasos.")
            print(f"  Alcanzado en t = {t_sync}.")

        print(f"  Parámetro de orden final:  r(T) = {r_final:.4f}  (= 1.0 indica sync perfecta)")

    elif tipo == "punto fijo no trivial":
        print("  ✘  LA SUBRED NO SINCRONIZÓ  (punto fijo no trivial)")
        print(sep2)
        print(f"  El sistema convergió a un estado fijo con fases distintas.")
        print(f"  Estado final: θ = {theta_final}")
        print(f"  r(0) = {r_ini:.4f}  →  r(T) = {r_final:.4f}")
        print("  Las fases quedaron bloqueadas sin consenso.")

    elif tipo == "ciclo":
        periodo = clasif["periodo"]
        t_inicio = clasif["tiempo"]
        print(f"  ✘  LA SUBRED NO SINCRONIZÓ  (ciclo de periodo {periodo})")
        print(sep2)
        print(f"  El sistema entró en un ciclo de periodo {periodo} a partir de t = {t_inicio}.")
        print(f"  Estado en t = {T}: θ = {theta_final}")
        print(f"  r(0) = {r_ini:.4f}  →  r(T) = {r_final:.4f}")
        print("  No hubo consenso de fases dentro del horizonte simulado.")

    else:  # transitorio
        print(f"  ?  RESULTADO INDETERMINADO  (transitorio en T = {T})")
        print(sep2)
        print(f"  No se detectó sincronización ni ciclo en {T} pasos.")
        print(f"  Estado en t = {T}: θ = {theta_final}")
        print(f"  r(0) = {r_ini:.4f}  →  r(T) = {r_final:.4f}")
        print("  Considera aumentar T o T_max para continuar la evolución.")

    print(sep)
    print()


def imprimir_conclusion_exhaustiva(res, N, M, kappa, tipo_grafo):
    """
    Conclusión resumida al final de la exploración exhaustiva:
    indica el régimen dominante y si la sincronización es mayoritaria.
    """
    sep  = "=" * 50
    sep2 = "-" * 50

    rho_sync = res["rho_sync"]
    rho_fix  = res["rho_fix"]
    rho_cyc  = res["rho_cyc"]
    total    = res["total_configuraciones"]

    # Régimen dominante
    regimenes = {
        "sincronización":          rho_sync,
        "punto fijo no trivial":   rho_fix,
        "ciclo":                   rho_cyc,
        "transitorio":             res["rho_transitorio"],
    }
    dominante = max(regimenes, key=regimenes.get)

    print()
    print(sep)
    print("  CONCLUSIÓN  (exploración exhaustiva)")
    print(sep)
    print(f"  Grafo: {tipo_grafo},  N={N},  M={M},  κ={kappa}")
    print(f"  Espacio total: {total} configuraciones")
    print(sep2)
    print(f"  Sincronización:          {res['sync']:>6}  ({rho_sync*100:5.1f} %)")
    print(f"  Punto fijo no trivial:   {res['fix']:>6}  ({rho_fix*100:5.1f} %)")
    print(f"  Ciclo:                   {res['cyc']:>6}  ({rho_cyc*100:5.1f} %)")
    print(f"  Transitorio:             {res['transitorio']:>6}  ({res['rho_transitorio']*100:5.1f} %)")
    print(sep2)

    if rho_sync >= 0.5:
        print(f"  ✔  La MAYORÍA de las condiciones iniciales SINCRONIZAN")
        print(f"     ({rho_sync*100:.1f} % del espacio de configuraciones).")
    elif rho_sync > 0:
        print(f"  ~  Solo una MINORÍA de condiciones iniciales sincronizan")
        print(f"     ({rho_sync*100:.1f} %).  Régimen dominante: {dominante}.")
    else:
        print(f"  ✘  NINGUNA condición inicial sincronizó con estos parámetros.")
        print(f"     Régimen dominante: {dominante}.")

    if res["tiempos_sync"]:
        t_med = float(np.mean(res["tiempos_sync"]))
        t_max = max(res["tiempos_sync"])
        print(f"     Tiempo medio de sync: {t_med:.1f} pasos  |  máximo: {t_max} pasos.")

    print(sep)
    print()


def imprimir_conclusion_barrido(df):
    """
    Conclusión del barrido de kappa: indica el kappa óptimo
    y describe la tendencia de sincronización.
    """
    sep  = "=" * 50
    sep2 = "-" * 50

    k_opt    = int(df.loc[df["rho_sync"].idxmax(), "kappa"])
    rho_max  = float(df["rho_sync"].max())
    k_min    = int(df["kappa"].min())
    k_max_df = int(df["kappa"].max())

    # ¿Monótona creciente, decreciente o con máximo interior?
    primera = float(df.iloc[0]["rho_sync"])
    ultima  = float(df.iloc[-1]["rho_sync"])

    print()
    print(sep)
    print("  CONCLUSIÓN  (barrido de κ)")
    print(sep)
    print(f"  Rango de κ explorado: {k_min} … {k_max_df}")
    print(sep2)

    if rho_max == 0.0:
        print("  ✘  No se observó sincronización en ningún valor de κ.")
    else:
        print(f"  ✔  Máxima sincronización en κ = {k_opt}  (ρ_sync = {rho_max*100:.1f} %)")

        if k_opt == k_min:
            print("     La sincronización es mayor para κ pequeño (umbral bajo).")
            print("     Aumentar κ reduce el acoplamiento efectivo.")
        elif k_opt == k_max_df:
            print("     La sincronización crece con κ hasta el extremo explorado.")
        else:
            print(f"     Existe un κ óptimo interior: κ = {k_opt}.")
            print("     Para κ < óptimo el acoplamiento es demasiado sensible;")
            print("     para κ > óptimo el umbral es demasiado restrictivo.")

        if ultima < 0.01:
            print(f"  ✘  Para κ = {k_max_df} la sincronización es nula:")
            print("     el umbral es tan alto que ningún vértice ajusta su fase.")

    print(sep)
    print()


# ============================================================
# 7. Interfaz con widgets
# ============================================================

N_widget = widgets.IntSlider(value=3, min=2, max=7, step=1, description="N")
M_widget = widgets.IntSlider(value=4, min=2, max=12, step=1, description="M")
kappa_widget = widgets.IntSlider(value=2, min=1, max=20, step=1, description="kappa")
omega_widget = widgets.IntSlider(value=0, min=0, max=11, step=1, description="omega*")
epsilon_widget = widgets.FloatSlider(value=1.0, min=0.1, max=6.0, step=0.1, description="epsilon")
T_widget = widgets.IntSlider(value=20, min=1, max=100, step=1, description="T")

grafo_widget = widgets.Dropdown(
    options=["Completo", "Camino", "Ciclo", "Estrella"],
    value="Completo", description="Grafo",
)
modo_theta_widget = widgets.Dropdown(
    options=["Ejemplo paper", "Aleatoria", "Manual simple"],
    value="Ejemplo paper", description="theta0",
)

# NUEVO: selector de convención epsilon para la subred
epsilon_modo_widget = widgets.ToggleButtons(
    options=[("Estricto  d < ε  (paper)", "estricto"),
             ("Incluyente  d ≤ ε  (didáctico)", "incluyente")],
    value="estricto",
    description="Subred:",
    style={"description_width": "initial"},
)

boton_simular    = widgets.Button(description="Simular trayectoria",   button_style="success")
boton_exhaustivo = widgets.Button(description="Exploración exhaustiva", button_style="warning")
boton_barrido    = widgets.Button(description="Barrido de kappa",       button_style="info")
salida = widgets.Output()


# ------------------------------------------------------------
# Actualización de rangos (orden correcto para ipywidgets)
# ------------------------------------------------------------

def actualizar_rangos(*args):
    N    = N_widget.value
    M    = M_widget.value
    tipo = grafo_widget.value

    omega_widget.max   = M - 1
    epsilon_widget.max = M // 2 + 1

    modelo_ref = KuramotoDiscreto(N=N, M=M, kappa=1, tipo_grafo=tipo)
    degmax     = max(dict(modelo_ref.G.degree()).values())
    kappa_max  = max(1, degmax * (M // 2) + 1)

    # Ajustar value ANTES de max para evitar error de validación en ipywidgets
    kappa_widget.value = min(kappa_widget.value, kappa_max)
    kappa_widget.max   = kappa_max


for w in [N_widget, M_widget, grafo_widget]:
    w.observe(actualizar_rangos, names="value")

actualizar_rangos()


# ------------------------------------------------------------
# Condición inicial
# ------------------------------------------------------------

def construir_theta0(N, M, modo):
    if modo == "Ejemplo paper":
        # Ejemplo canónico del paper: theta0 = (0, M//2, M-1, 0, 0, ...)
        # Los nodos más allá del tercero se fijan en 0.
        base  = [0, M // 2, M - 1]
        theta = (base + [0] * N)[:N]
        return np.array(theta, dtype=int) % M
    elif modo == "Manual simple":
        return np.array([i % M for i in range(N)], dtype=int)
    else:
        return np.random.randint(0, M, size=N)


# ------------------------------------------------------------
# Acción: simulación individual
# ------------------------------------------------------------

def on_simular_clicked(b):
    with salida:
        clear_output(wait=True)

        N    = N_widget.value
        M    = M_widget.value
        kappa = kappa_widget.value
        omega = omega_widget.value
        epsilon = epsilon_widget.value
        T    = T_widget.value
        tipo_grafo   = grafo_widget.value
        modo         = modo_theta_widget.value
        epsilon_modo = epsilon_modo_widget.value

        modelo = KuramotoDiscreto(
            N=N, M=M, kappa=kappa, omega=omega,
            tipo_grafo=tipo_grafo, epsilon=epsilon, epsilon_modo=epsilon_modo,
        )
        theta0   = construir_theta0(N, M, modo)
        resultado = modelo.simular(theta0, T=T)
        clasif    = modelo.clasificar_atractor(theta0, T_max=max(200, T))
        tabla     = modelo.tabla_trayectoria(resultado)

        print("SIMULACIÓN INDIVIDUAL")
        print("=" * 40)
        print(f"Grafo:        {tipo_grafo}")
        print(f"N={N},  M={M},  kappa={kappa},  omega*={omega}")
        print(f"epsilon={epsilon}  ({epsilon_modo})")
        print(f"theta0 = {tuple(theta0)}")
        print()
        print(f"Clasificación: {clasif['tipo']}")
        print(f"Tiempo:        {clasif['tiempo']}")
        print(f"Periodo:       {clasif['periodo']}")
        print(f"Estado final:  {tuple(clasif['estado_final'])}")
        print()

        plot_heatmap(resultado)
        plot_order_parameter(resultado)

        mitad = len(resultado["trayectoria"]) // 2
        plot_grafo(modelo, resultado["trayectoria"][0],     title="Snapshot inicial")
        plot_grafo(modelo, resultado["trayectoria"][mitad], title=f"Snapshot t={mitad}")
        plot_grafo(modelo, resultado["trayectoria"][-1],    title=f"Snapshot final t={T}")

        display(tabla)
        imprimir_conclusion_simulacion(modelo, resultado, clasif, T)


# ------------------------------------------------------------
# Acción: exploración exhaustiva
# ------------------------------------------------------------

def on_exhaustivo_clicked(b):
    with salida:
        clear_output(wait=True)

        N    = N_widget.value
        M    = M_widget.value
        kappa = kappa_widget.value
        omega = omega_widget.value
        epsilon = epsilon_widget.value
        tipo_grafo   = grafo_widget.value
        epsilon_modo = epsilon_modo_widget.value
        total = M ** N

        print("EXPLORACIÓN EXHAUSTIVA")
        print("=" * 40)
        print(f"Total de configuraciones: M^N = {M}^{N} = {total}")
        print()

        if total > 20_000:
            print("Advertencia: espacio demasiado grande. Reduce N o M.")
            return

        res = exploracion_exhaustiva(
            N=N, M=M, kappa=kappa, omega=omega,
            tipo_grafo=tipo_grafo, epsilon=epsilon,
            epsilon_modo=epsilon_modo, T_max=150,
        )

        display(pd.DataFrame([{
            "total":       res["total_configuraciones"],
            "sync":        res["sync"],
            "fix":         res["fix"],
            "cyc":         res["cyc"],
            "transitorio": res["transitorio"],
            "rho_sync":    res["rho_sync"],
            "rho_fix":     res["rho_fix"],
            "rho_cyc":     res["rho_cyc"],
            "rho_trans":   res["rho_transitorio"],
            "subredes":    res["num_subredes_realizables"],
            "transiciones": res["num_transiciones"],
            "Bell_BN":     res["Bell_BN"],
        }]))

        plot_hist_tiempos(res["tiempos_sync"])
        plot_diagrama_transiciones(res["transiciones"])
        imprimir_conclusion_exhaustiva(res, N, M, kappa, tipo_grafo)


# ------------------------------------------------------------
# Acción: barrido de kappa
# ------------------------------------------------------------

def on_barrido_clicked(b):
    with salida:
        clear_output(wait=True)

        N    = N_widget.value
        M    = M_widget.value
        omega = omega_widget.value
        epsilon = epsilon_widget.value
        tipo_grafo   = grafo_widget.value
        epsilon_modo = epsilon_modo_widget.value
        total = M ** N

        print("BARRIDO DE KAPPA")
        print("=" * 40)
        print(f"Configuraciones por kappa: {total}")
        print()

        if total > 10_000:
            print("Advertencia: barrido pesado. Reduce N o M antes de ejecutar.")
            return

        df = barrido_kappa(
            N=N, M=M, omega=omega,
            tipo_grafo=tipo_grafo, epsilon=epsilon,
            epsilon_modo=epsilon_modo, T_max=150,
        )

        display(df)
        plot_mapa_regimenes(df)
        if len(df) > 0:
            imprimir_conclusion_barrido(df)


boton_simular.on_click(on_simular_clicked)
boton_exhaustivo.on_click(on_exhaustivo_clicked)
boton_barrido.on_click(on_barrido_clicked)


# ============================================================
# 7. Panel principal
# ============================================================

panel = widgets.VBox([
    widgets.HTML("<h2>Simulador de Sincronización Discreta tipo Kuramoto sobre Grafos</h2>"),
    widgets.HTML("""
    <p>
    Modelo: <b>F<sub>κ</sub>(θ)<sub>i</sub> = θ<sub>i</sub> + ω* + σ<sub>κ</sub>(S<sub>i</sub>(θ)) mod M</b>
    </p>
    <p>
    <b>Correcciones aplicadas:</b>
    σ<sub>κ</sub>(S) = +1 si S ≥ κ, −1 si S ≤ −κ, 0 si |S| &lt; κ  (umbral incluyente).<br>
    d<sub>M</sub>(a,b) = ((a−b+⌊M/2⌋) mod M) − ⌊M/2⌋  (orden natural).<br>
    Subred ε-sincronizada: modo estricto d &lt; ε (paper) o incluyente d ≤ ε (didáctico).
    </p>
    """),
    widgets.HBox([N_widget, M_widget, kappa_widget]),
    widgets.HBox([omega_widget, epsilon_widget, T_widget]),
    widgets.HBox([grafo_widget, modo_theta_widget]),
    epsilon_modo_widget,
    widgets.HBox([boton_simular, boton_exhaustivo, boton_barrido]),
    salida,
])

display(panel)


# ============================================================
# 8. Verificación rápida del ejemplo del paper
# ============================================================
#
# Ejecuta esto en una celda aparte para confirmar la corrección 1:
#
#   modelo = KuramotoDiscreto(N=3, M=4, kappa=2, omega=0,
#                             tipo_grafo="Completo", epsilon=1,
#                             epsilon_modo="estricto")
#   theta0 = [0, 2, 3]
#   resultado = modelo.simular(theta0, T=5)
#   print(resultado["trayectoria"])
#   # Esperado: (0,2,3) -> (3,2,3) -> (3,3,3)  [sincronización en t=2]
#   print(modelo.clasificar_atractor(theta0))
#   # Esperado: tipo='sync', tiempo=2, periodo=1
#   display(modelo.tabla_trayectoria(resultado))